In [8]:
import sys
import time
import importlib.util
import numpy as np
import pandas as pd
import pyfeng as pf
module_name = "pyfeng.sv_cos"
file_path = "./pyfeng/sv_cos.py"
spec = importlib.util.spec_from_file_location("pyfeng.sv_cos", "./pyfeng/sv_cos.py")
sv_cos = importlib.util.module_from_spec(spec)
sys.modules["pyfeng.sv_cos"] = sv_cos
spec.loader.exec_module(sv_cos)

In [9]:
# Standard Heston parameters
spot = 100.0
strike = np.array([80, 90, 100, 110, 120])
texp = 1.0

# PyFENG SvABC parameter convention:
sigma = 0.04    # initial variance (v0)
vov = 0.5       # vol-of-vol (eta)
mr = 1.5        # mean-reversion (kappa)
rho = -0.7      # correlation
theta = 0.04    # long-run variance (ubar)
intr = 0.03     # interest rate
divr = 0.01     # dividend yield

In [ ]:
# 1. Initialize the PyFeng original FFT Pricer
m_fft = pf.HestonFft(sigma, vov=vov, mr=mr, rho=rho, theta=theta, intr=intr, divr=divr)

# 2. Initialize our new COS Pricer
m_cos = sv_cos.HestonCos(sigma, vov=vov, mr=mr, rho=rho, theta=theta, intr=intr, divr=divr)

In [ ]:
# Price using FFT
t0 = time.perf_counter()
price_fft = m_fft.price(strike, spot, texp)
time_fft = (time.perf_counter() - t0) * 1000

# Price using COS
t0 = time.perf_counter()
price_cos = m_cos.price(strike, spot, texp)
time_cos = (time.perf_counter() - t0) * 1000

results = pd.DataFrame({
    "Strike": strike,
    "Prof's FFT Price": price_fft,
    "Our COS Price": price_cos,
    "Absolute Diff": np.abs(price_fft - price_cos)
})

print(f"FFT Time: {time_fft:.3f} ms")
print(f"COS Time: {time_cos:.3f} ms")
print("-" * 45)
display(results)

FFT Time: 3.949 ms
COS Time: 0.904 ms
---------------------------------------------


,Strike,Prof's FFT Price,Our COS Price,Absolute Diff
0,80,23.006535,23.006535,2.974472e-07
1,90,14.928948,14.928948,8.368403e-08
2,100,8.113489,8.113490,6.604023e-07
3,110,3.306261,3.306261,3.755554e-08
4,120,0.956587,0.956586,6.859967e-07
